# DwoPP for IP102 - Lifelong Object Re-Identification

This notebook trains the DwoPP model on IP102 dataset for continual/lifelong learning.

**Paper**: Positive Pair Distillation Considered Harmful: Continual Meta Metric Learning for Lifelong Object Re-Identification (BMVC 2022)

**Dataset**: IP102 - 25 pest classes, 4 tasks (7/6/6/6 split)

**GitHub**: https://github.com/nta2112/DWoPP-for-IP102

In [1]:
import os
import sys
import subprocess

# Clone repository
REPO_URL = os.environ.get('IP102_CODE_REPO', 'https://github.com/nta2112/DWoPP-for-IP102.git')
REPO_DIR = 'DWoPP-for-IP102'

if not os.path.exists(REPO_DIR):
    print(f'Cloning from {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repository already exists, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')

Cloning from https://github.com/nta2112/DWoPP-for-IP102.git...


Cloning into 'DWoPP-for-IP102'...


Working directory: /kaggle/working/DWoPP-for-IP102


In [2]:
# Install requirements
!pip install -q -r requirements.txt 2>/dev/null | tail -5

In [3]:
# Verify dataset location (optional - run_train will auto-discover)
import glob
import os

possible_paths = [
    '/kaggle/input/ip102',
    '/kaggle/input/ip102-dataset',
    '/kaggle/input/ip102-dataset/ip102',
]

data_root = None
for p in possible_paths:
    if os.path.exists(p) and any(f.endswith('.json') for f in os.listdir(p) if os.path.isfile(os.path.join(p, f))):
        data_root = p
        break

if data_root is None:
    for p in possible_paths:
        for root, dirs, files in os.walk(p):
            if any(f.endswith('.json') for f in files):
                data_root = root
                break
        if data_root:
            break

print(f'Data root: {data_root or "Not found (script will auto-discover)"}')
if data_root and os.path.exists(data_root):
    print(f'Files: {os.listdir(data_root)}')

Data root: Not found (script will auto-discover)


In [4]:
# Auto-detect GPUs
import torch

device_count = torch.cuda.device_count()
gpu_ids = list(range(device_count))
print(f'Available GPUs: {device_count}')
print(f'GPU IDs: {gpu_ids}')

gpu_arg = ','.join(map(str, gpu_ids)) if gpu_ids else '0'
print(f'GPU arg: {gpu_arg}')

Available GPUs: 2
GPU IDs: [0, 1]
GPU arg: 0,1


In [5]:
def run_train(model='DwoPP', max_tasks=0, memory_size=0, seed=0, epochs=200):    """    Run training for IP102 dataset.        Args:        model: Model name (DwoPP, DwPP, FT)        max_tasks: Maximum tasks to run (0 = all 4 tasks, 1 = quick test)        memory_size: Memory buffer size (not used in DwoPP)        seed: Random seed        epochs: Number of epochs per task    """    import subprocess    import sys    import os        if max_tasks == 0:        max_tasks = 4        task_num = min(max_tasks, 4)        # Find filtered_class.txt and classes.txt    filtered_class_path = None    classes_txt_path = None    dataset_root = None        # Search for dataset files    search_paths = [        '/kaggle/input/ip102',        '/kaggle/input/ip102-dataset',        '/kaggle/input/ip102-dataset/ip102',    ]    if 'data_root' in globals() and data_root and os.path.exists(data_root):        search_paths.insert(0, data_root)    search_paths = [p for p in search_paths if p and os.path.exists(p)]        for p in search_paths:        for root, dirs, files in os.walk(p):            if 'filtered_class.txt' in files:                filtered_class_path = os.path.join(root, 'filtered_class.txt')            if 'classes.txt' in files:                classes_txt_path = os.path.join(root, 'classes.txt')            if any(f.endswith('.json') for f in files):                if dataset_root is None:                    dataset_root = root        if filtered_class_path and classes_txt_path and dataset_root:            break        cmd = [        sys.executable, 'CL_train_DwoPP.py',        '--dataset', 'ip102',        '--exp_root', f'dmml/ip102_{model}_seed{seed}',        '--lr', '1e-4',        '--num_epochs', str(epochs),        '--lr_decay_start_epoch', '0',        '--weight_decay', '1e-4',        '--num_classes', '6',        '--distance_mode', 'hard_mining',        '--num_support', '5',        '--num_query', '1',        '--margin', '0.2',        '--img_height', '256',        '--img_width', '128',        '--num_workers', '4',        '--gpu', gpu_arg,        '--random_erasing',        '--remove_downsample',        '--cuda',        '--method', f'{model}_seed_{seed}',        '--loss_type', 'dmml',        '--preprocess_data_path', 'preprocess_dataset/',        '--start_task_id', '0',        '--weight_knowledge_distill', '1.0',        '--dmml_dist_metric', 'cosine',        '--distillation_dist_metric', 'cosine',        '--manual_seed', str(seed),        '--temperature', '1.0',        '--remove_positive_pair',    ]        # Only pass dataset_root if found; otherwise script will auto-discover    if dataset_root:        cmd.extend(['--dataset_root', dataset_root])    if filtered_class_path:        cmd.extend(['--filtered_class_path', filtered_class_path])    if classes_txt_path:        cmd.extend(['--classes_txt_path', classes_txt_path])        print(f'Running: {" ".join(cmd)}')    result = subprocess.run(cmd, capture_output=False)    return result.returncode == 0

In [6]:
# Quick test run (1 epoch per task for verification)
print("=== QUICK TEST RUN (1 epoch per task) ===")
success = run_train(model='DwoPP', max_tasks=0, seed=0, epochs=1)
print(f'Quick test: {"PASSED" if success else "FAILED"}')

=== QUICK TEST RUN (1 epoch per task) ===
Running: /usr/bin/python3 CL_train_DwoPP.py --dataset ip102 --exp_root dmml/ip102_DwoPP_seed0 --lr 2e-4 --num_epochs 1 --lr_decay_start_epoch 0 --weight_decay 1e-4 --num_classes 16 --distance_mode hard_mining --num_support 5 --num_query 1 --margin 0.4 --img_height 256 --img_width 128 --num_workers 4 --gpu 0,1 --random_erasing --remove_downsample --cuda --method DwoPP_seed_0 --loss_type dmml --preprocess_data_path preprocess_dataset/ --start_task_id 0 --weight_knowledge_distill 1.0 --dmml_dist_metric euclidean --distillation_dist_metric euclidean --manual_seed 0 --temperature 1.0 --remove_positive_pair
Dataset root: /kaggle/input/datasets/nta212/ip102-for-object-detection
Filtered classes: /kaggle/input/datasets/nta212/ip102-for-object-detection/filtered_class.txt
Classes txt: /kaggle/input/datasets/nta212/ip102-for-object-detection/classes.txt
Dataset: ip102
Model: ResNet-50
Optimizer: Adam
Image height: 256
Image width: 128
Loss: dmml
  margin

100%|██████████| 97.8M/97.8M [00:00<00:00, 211MB/s]


no bnneck
Loaded 25 filtered classes from /kaggle/input/datasets/nta212/ip102-for-object-detection/filtered_class.txt
Loaded 102 class names from /kaggle/input/datasets/nta212/ip102-for-object-detection/classes.txt
Task 0: 7 classes, 2873 images
Task 1: 6 classes, 1283 images
Task 2: 6 classes, 1864 images
Task 3: 6 classes, 3525 images
Saved preprocessed splits to preprocess_dataset/ip102/ip102_task_splits.pkl
Load task 0 train data: 2873 samples, 7 classes
=== Epoch 0/1 ===
=====> lr adjusted to 0.000001
Average loss: nan, dmml_losses: nan, know_distill_losses: nan
Time elapsed: 0h 0m
Final model saved.
Loaded 25 filtered classes from /kaggle/input/datasets/nta212/ip102-for-object-detection/filtered_class.txt
Loaded 102 class names from /kaggle/input/datasets/nta212/ip102-for-object-detection/classes.txt
Load val data: 2176 images
Loaded 25 filtered classes from /kaggle/input/datasets/nta212/ip102-for-object-detection/filtered_class.txt
Loaded 102 class names from /kaggle/input/datas

In [7]:
# Full training run
# print("=== FULL TRAINING RUN ===")
# success = run_train(model='DwoPP', max_tasks=0, seed=0, epochs=200)
# print(f'Full training: {"COMPLETED" if success else "FAILED"}')

In [8]:
# Display results
import pandas as pd
import glob

result_files = glob.glob('dmml/ip102_DwoPP_seed_*/results.csv')
if not result_files:
    result_files = glob.glob('**/results.csv', recursive=True)

print(f'Found result files: {result_files}')

for rf in result_files:
    print(f'\n--- {rf} ---')
    df = pd.read_csv(rf)
    print(df.to_string(index=False))

Found result files: ['dmml/ip102_DwoPP_seed0/DwoPP_seed_0/results.csv', 'dmml/market1501/test_run/results.csv']

--- dmml/ip102_DwoPP_seed0/DwoPP_seed_0/results.csv ---
 task  numclass  cnn_top1  nme_top1    R@1    R@5   R@10    mAP    AUROC    FPR95  Plasticity  Forgetting  Overall
    0         7       0.0       0.0 0.5724 0.8083 0.8854 0.1649      NaN      NaN      0.0000         0.0   0.1649
    1        13       0.0       0.0 0.5724 0.8083 0.8854 0.1649      NaN      NaN      0.1649         0.0   0.1649
    2        19       0.0       0.0 0.5724 0.8083 0.8854 0.1649 0.746283 0.628079      0.1649         0.0   0.1649
    3        25       0.0       0.0 0.5724 0.8083 0.8854 0.1649 0.613621 0.765537      0.1649         0.0   0.1649

--- dmml/market1501/test_run/results.csv ---
Empty DataFrame
Columns: [task, numclass, cnn_top1, nme_top1, R@1, R@5, R@10, mAP, AUROC, FPR95, Plasticity, Forgetting, Overall]
Index: []


In [9]:
# Also check history.json
import json
import glob

history_files = glob.glob('**/history.json', recursive=True)
for hf in history_files:
    print(f'\n--- {hf} ---')
    with open(hf) as f:
        history = json.load(f)
    for entry in history:
        print(entry)


--- dmml/ip102_DwoPP_seed0/DwoPP_seed_0/history.json ---
{'task': 0, 'numclass': 7, 'cnn_top1': 0.0, 'nme_top1': 0.0, 'R@1': 0.572429045337265, 'R@5': 0.8083302617029119, 'R@10': 0.8853667526723185, 'mAP': 0.16493368758833388, 'AUROC': None, 'FPR95': None, 'Plasticity': 0.0, 'Forgetting': 0.0, 'Overall': 0.16493368758833388}
{'task': 1, 'numclass': 13, 'cnn_top1': 0.0, 'nme_top1': 0.0, 'R@1': 0.572429045337265, 'R@5': 0.8083302617029119, 'R@10': 0.8853667526723185, 'mAP': 0.16493368758833388, 'AUROC': None, 'FPR95': None, 'Plasticity': 0.16493368758833388, 'Forgetting': 0.0, 'Overall': 0.16493368758833388}
{'task': 2, 'numclass': 19, 'cnn_top1': 0.0, 'nme_top1': 0.0, 'R@1': 0.572429045337265, 'R@5': 0.8083302617029119, 'R@10': 0.8853667526723185, 'mAP': 0.16493368758833388, 'AUROC': 0.7462829981999526, 'FPR95': 0.6280788177339901, 'Plasticity': 0.16493368758833388, 'Forgetting': 0.0, 'Overall': 0.16493368758833388}
{'task': 3, 'numclass': 25, 'cnn_top1': 0.0, 'nme_top1': 0.0, 'R@1': 0